<a href="https://colab.research.google.com/github/Chintalasiri/OIBSIP/blob/main/SiriChintala_Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DataAnalytics_Project/Datasets/OnlineRetail.csv", encoding='latin1')

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df = df.dropna(subset=['Description'])

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=['CustomerID'])

In [ ]:
df.isnull().sum()

## Data Cleaning

- Missing values were identified in the Description and CustomerID columns.
- Rows with missing values were removed to ensure accurate customer segmentation.
- The cleaned dataset was used for further analysis.

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

## Data Cleaning - Duplicate Records

- The dataset initially contained 5,225 duplicate records.
- Duplicate rows were removed using the `drop_duplicates()` function.
- Removing duplicates improves the quality of the dataset and ensures accurate customer segmentation.

 Descriptive statistics: average purchase value, purchase frequency, customer
lifetime value

In [ ]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [ ]:
average_purchase = df['TotalAmount'].mean()
print("Average Purchase Value:", average_purchase)

In [ ]:
purchase_frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().mean()

print("Average Purchase Frequency:", purchase_frequency)

In [ ]:
customer_lifetime_value = df.groupby('CustomerID')['TotalAmount'].sum().mean()

print("Average Customer Lifetime Value:", customer_lifetime_value)

# Descriptive Statistics

The following customer metrics were calculated:

- Average Purchase Value
- Purchase Frequency
- Customer Lifetime Value

These metrics help understand customer purchasing behaviour before performing clustering.

convert innovoiceDate

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

In [ ]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [ ]:
rfm.head()

In [ ]:
rfm

# Feature Selection - RFM Analysis

Three important behavioural features were selected for customer segmentation:

- **Recency (R):** Number of days since the customer's last purchase.
- **Frequency (F):** Number of unique purchases made by the customer.
- **Monetary (M):** Total amount spent by the customer.

These RFM features help group customers with similar purchasing behaviour before applying K-Means clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
rfm_scaled = scaler.fit_transform(rfm)

# Data Normalisation

Before applying K-Means clustering, the selected RFM features were standardised using **StandardScaler**.

This ensures that Recency, Frequency, and Monetary contribute equally during clustering and prevents features with larger values from dominating the results.

Apply Kmeans

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss = []

In [ ]:
for i in range(1,11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)  # Add n_init to suppress warning
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1,11), wcss, marker='o')

plt.title("Elbow Method")

plt.xlabel("Number of Clusters")

plt.ylabel("WCSS")

plt.show()

# Elbow Method

The Elbow Method was used to determine the optimal number of clusters for K-Means clustering.

The point where the graph forms an "elbow" indicates the most suitable value of K. This value provides a balance between clustering accuracy and simplicity.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)

In [ ]:
clusters = kmeans.fit_predict(rfm_scaled)

In [ ]:
rfm['Cluster'] = clusters

In [ ]:
rfm.head()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=rfm,
    x='Recency',
    y='Monetary',
    hue='Cluster',
    palette='Set2'
)

plt.title("Customer Segments: Recency vs Monetary")
plt.show()

### Observation

This scatter plot shows customer segments based on Recency and Monetary value. Customers with similar purchasing behaviour are grouped into different clusters using K-Means.

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=rfm,
    x='Frequency',
    y='Monetary',
    hue='Cluster',
    palette='Set2'
)

plt.title("Customer Segments: Frequency vs Monetary")
plt.xlabel("Frequency")
plt.ylabel("Monetary")

plt.show()

### Observation

This scatter plot shows customer segments based on Frequency and Monetary values.

Customers who purchase more frequently and spend more are grouped together, while customers with lower purchase frequency and lower spending fall into different clusters.

In [ ]:
cluster_profile = rfm.groupby('Cluster').mean()

cluster_profile

## Cluster Profile

The average Recency, Frequency, and Monetary values were calculated for each cluster.

This helps understand the characteristics of each customer segment.

In [ ]:
cluster_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()

cluster_profile

## Cluster Interpretation

### Cluster 0 – Loyal Customers
- Low Recency (customers purchased recently)
- High Frequency
- High Monetary value

These customers purchase regularly and spend a significant amount. They are loyal customers and should be rewarded with loyalty programs and exclusive offers.

---

### Cluster 1 – Inactive Customers
- Very High Recency
- Very Low Frequency
- Very Low Monetary value

These customers have not purchased for a long time and spend very little. Businesses can target them with re-engagement campaigns and special discounts.

---

### Cluster 2 – Premium Customers
- Lowest Recency
- Highest Frequency
- Highest Monetary value

These are the most valuable customers. They purchase very frequently and spend the most money. Businesses should focus on retaining them with VIP memberships and personalized offers.

---

### Cluster 3 – Regular Customers
- Moderate Recency
- Moderate Frequency
- Moderate Monetary value

These customers purchase occasionally and have average spending. Businesses can encourage them to purchase more through promotional campaigns.

In [ ]:
rfm['Cluster'].value_counts()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=rfm, x='Cluster', palette='Set2')

plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")

plt.show()

## Observation

The bar chart shows the number of customers in each cluster.

It helps identify which customer segment has the highest and lowest number of customers. This information is useful for planning targeted marketing strategies.

# Conclusion

This project successfully applied K-Means clustering to segment customers based on their purchasing behaviour using the RFM (Recency, Frequency, Monetary) model.

Key findings:
- Premium customers contribute the highest revenue and should be retained through VIP programs.
- Loyal customers can be rewarded with exclusive offers to maintain engagement.
- Regular customers can be encouraged to purchase more through promotions.
- Inactive customers require re-engagement campaigns to increase their activity.

Customer segmentation helps businesses create targeted marketing strategies, improve customer satisfaction, and maximize overall revenue.